# Project0 Colab Tutorial

This notebook prepares a Google Colab GPU runtime, installs Project0 and its dependencies, starts a local large language model through Ollama, and makes the Project0 Dashboard available through a temporary public link.

## What you will learn

By completing the notebook, you will see how the main parts of the Colab environment work together:

- **Google Colab** supplies the temporary Linux computer and NVIDIA GPU.
- **Ollama** manages and runs the local language model.
- **`qwen2.5:7b`** is the language model used for Project0 reasoning in this notebook.
- **Project0** provides the Dashboard and agent workflows.
- **Cloudflare Quick Tunnel** gives your browser a temporary HTTPS link to the Dashboard running inside Colab.

The request path is:

```text
Your browser → Cloudflare Quick Tunnel → Project0 Dashboard → Ollama → qwen2.5:7b
```

## Before you begin

In Colab, select **Runtime → Change runtime type**, choose an available **GPU** accelerator, and save the setting. Then run the notebook from top to bottom. You may use **Runtime → Run all**; the optional cleanup in Step 8 is disabled by default so the Dashboard remains available.

Colab runtimes are temporary. Installed software, downloaded models, logs, and other files under `/content` disappear when the runtime is deleted.

## Step 1 - Clone or Update Project0

This step retrieves the Project0 source code from its public GitHub repository and makes it the notebook's working directory.

- If `/content/project0` does not exist, the repository is cloned.
- If it already exists, `git pull --ff-only` downloads newer commits without creating a merge commit.
- No GitHub account, access token, or Colab secret is required for the public repository.

When the step succeeds, the final message is `Project0 repository ready.` and subsequent cells operate from `/content/project0`.

If the update cannot proceed, the existing checkout may contain local changes or may no longer have a direct update path. Review the preceding Git message before rerunning the cell.

In [ ]:
from pathlib import Path
import subprocess

REPO_DIR = Path("/content/project0")
REPO_URL = "https://github.com/pgailinas/project0.git"

if not REPO_DIR.exists():
    print("Cloning Project0...")
    subprocess.run(
        ["git", "clone", REPO_URL, str(REPO_DIR)],
        check=True,
    )
else:
    print("Project0 already exists. Updating repository...")
    subprocess.run(
        ["git", "-C", str(REPO_DIR), "pull", "--ff-only"],
        check=True,
    )

%cd /content/project0

print("Project0 repository ready.")



## Step 2 - Prepare the Environment and Install Project0

This step first confirms that the Colab runtime uses Python 3.12 or newer, which Project0 requires.

Google Colab includes the optional `jieba` language-segmentation package. Project0 does not require this package, but MkDocs Material automatically detects and imports it when present. Under Python 3.13, that import produces compatibility warnings during Documentation Agent validation. The cell removes `jieba` to keep the validation results focused on Project0 documentation.

The cell then installs Project0 and its Python dependencies into the current runtime. The editable installation points Python to the source files in `/content/project0`; it does not replace the cloned source with a separate packaged copy. This is useful during testing because the installed application follows the checked-out repository.

Dependency installation can take a few minutes during a new Colab session. The step is complete when `Project0 installation complete.` appears. A Python-version error means the selected Colab runtime is not compatible with the current Project0 requirements.

The cell is safe to rerun. If `jieba` has already been removed, the uninstall command reports that it is not installed and continues.

In [ ]:
import sys
import subprocess

print("Python:", sys.version)

if sys.version_info < (3, 12):
    raise RuntimeError(
        "Project0 requires Python >= 3.12. "
        f"This Colab runtime is Python "
        f"{sys.version_info.major}.{sys.version_info.minor}."
    )

# Colab includes jieba even though Project0 does not require it.
# MkDocs Material detects the optional package and imports it,
# producing Python 3.13 compatibility warnings during validation.
print("\nRemoving unused Colab jieba package...")

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "uninstall",
        "-y",
        "jieba",
    ],
    check=False,
)

print("\nInstalling Project0 from the cloned repository...")

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-e",
        "/content/project0",
    ],
    check=True,
)

print("\nProject0 installation complete.")



## Step 3 - Verify the Colab GPU

Large language models perform their calculations much faster on a supported graphics processor. This step asks PyTorch whether CUDA—the NVIDIA GPU computing interface—is available, then reports the assigned GPU and its total video memory (VRAM).

Expected results include:

- `CUDA available: True`
- A GPU model name
- A positive VRAM value

If CUDA is unavailable, select **Runtime → Change runtime type → GPU** and restart the notebook. The exact GPU model can vary between Colab sessions and account types.

In [ ]:
import torch

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if not torch.cuda.is_available():
    raise RuntimeError(
        "No CUDA GPU is available. "
        "In Colab, select Runtime > Change runtime type > GPU."
    )

gpu_index = 0
gpu_name = torch.cuda.get_device_name(gpu_index)
gpu_props = torch.cuda.get_device_properties(gpu_index)
total_vram_gb = gpu_props.total_memory / (1024 ** 3)

print("GPU:", gpu_name)
print(f"VRAM: {total_vram_gb:.1f} GB")

## Step 4 - Install and Start Ollama

Ollama is the local model runtime used by this notebook. It downloads, loads, and executes language models and exposes a local API that Project0 can call. It is not the language model itself.

This step installs the `zstd` compression dependency and Ollama, starts `ollama serve` as a background process, and waits for its API at `127.0.0.1:11434`. That address is private to the Colab runtime and is not exposed directly to the Internet. Ollama output is recorded in `/content/ollama.log`.

Successful output includes `Ollama server: RUNNING` and an HTTP status of `200`. If Ollama does not become ready, the cell reports its startup log to help identify the failure.

(Expect ~27 seconds for 1st run of this cell)

In [ ]:
!apt-get update -qq
!apt-get install -y zstd
!curl -fsSL https://ollama.com/install.sh | sh

import subprocess
import time
import urllib.request

OLLAMA_LOG = "/content/ollama.log"

ollama_server = subprocess.Popen(
    ["ollama", "serve"],
    stdout=open(OLLAMA_LOG, "w"),
    stderr=subprocess.STDOUT,
)

print(f"Ollama server started (PID {ollama_server.pid}).")

ollama_ready = False
for attempt in range(15):
    time.sleep(1)
    try:
        response = urllib.request.urlopen(
            "http://127.0.0.1:11434/api/tags",
            timeout=5,
        )
        print("Ollama server: RUNNING")
        print("HTTP status:", response.status)
        ollama_ready = True
        break
    except Exception:
        pass

if not ollama_ready:
    raise RuntimeError(
        "Ollama server did not start.\n\n" + open(OLLAMA_LOG).read()
    )

## Step 5 - Load and Verify `qwen2.5:7b`

This step downloads the `qwen2.5:7b` model into Ollama. The name identifies the Qwen 2.5 model family and its approximately seven-billion-parameter size. The initial download can take several minutes, but rerunning the cell in the same Colab runtime can reuse the downloaded model.

The cell then:

1. Lists the models available to Ollama.
2. Sends a short test prompt to verify inference.
3. Displays active model processing information.

The expected test response is `Ollama GPU test successful.` After the inference, inspect the `PROCESSOR` column produced by `ollama ps`; it should indicate GPU use. A CPU assignment will usually work much more slowly.

(Expect ~1 minute 49 seconds for 1st run of this cell)

In [ ]:
!ollama pull qwen2.5:7b
!ollama list
!ollama run qwen2.5:7b "Reply with exactly: Ollama GPU test successful."
!ollama ps

## Step 6 - Configure and Start Project0

This step configures Project0 and starts the Dashboard as a background process. It creates a unique run identifier, prepares a directory for diagnostic artifacts, stops any Dashboard process left by an earlier execution, and clears the previous Dashboard log.

The notebook configures:

- Ollama as the reasoning provider.
- `qwen2.5:7b` as the Documentation Agent reasoning model.
- OpenAlex, Crossref, arXiv, and OpenReview as Research Agent sources.
- DEBUG logging for development and diagnostics.

Project0 listens inside Colab at `http://127.0.0.1:8001`. The cell waits up to 30 seconds for that address to return a successful response. It reports `Project0 Dashboard: RUNNING` only after confirming readiness. Runtime messages are written to `/content/project0_dashboard.log`.

If the Dashboard process exits early or does not become ready before the timeout, the cell prints the Dashboard log, stops any remaining process, and raises an error. Review the reported diagnostics, correct the problem, and rerun this step. Continue to the public-link step only after the Dashboard is running successfully.

In [ ]:
import os
import subprocess
import sys
import time
import urllib.request
from datetime import datetime
from pathlib import Path

# ============================================================
# Project0 Colab Debug Configuration
# ============================================================

PROJECT0_ROOT = Path("/content/project0")
PROJECT0_LOG = Path("/content/project0_dashboard.log")
DASHBOARD_URL = "http://127.0.0.1:8001"
DASHBOARD_STARTUP_TIMEOUT_SECONDS = 30

DEBUG_ARTIFACT_ROOT = Path(
    "/content/project0_debug_artifacts"
)
DEBUG_ARTIFACT_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S")

print(f"Project0 debug iteration: {RUN_ID}")


# ============================================================
# Stop any previous Project0 Dashboard process.
# ============================================================

subprocess.run(
    [
        "pkill",
        "-f",
        "project0.dashboard.dashboard_app",
    ],
    check=False,
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
)

time.sleep(1)


# ============================================================
# Clear the Dashboard log.
# ============================================================

PROJECT0_LOG.write_text("")


# ============================================================
# Project0 runtime configuration.
# ============================================================

os.environ["PROJECT0_REASONING_PROVIDER"] = "ollama"
os.environ["PROJECT0_DOCUMENTATION_OLLAMA_MODEL"] = (
    "qwen2.5:7b"
)
os.environ["PROJECT0_RESEARCH_SOURCE_PROVIDERS"] = (
    "openalex,crossref,arxiv,openreview"
)
os.environ["PROJECT0_LOG_LEVEL"] = "DEBUG"

print("\nProject0 configuration:")
print(
    "  Reasoning provider:",
    os.environ["PROJECT0_REASONING_PROVIDER"],
)
print(
    "  Ollama model:",
    os.environ["PROJECT0_DOCUMENTATION_OLLAMA_MODEL"],
)
print(
    "  Research sources:",
    os.environ["PROJECT0_RESEARCH_SOURCE_PROVIDERS"],
)
print(
    "  Log level:",
    os.environ["PROJECT0_LOG_LEVEL"],
)


# ============================================================
# Start the Project0 Dashboard.
# ============================================================

with PROJECT0_LOG.open(
    "w",
    buffering=1,
) as log_file:
    dashboard = subprocess.Popen(
        [
            sys.executable,
            "-m",
            "project0.dashboard.dashboard_app",
        ],
        cwd=PROJECT0_ROOT,
        stdout=log_file,
        stderr=subprocess.STDOUT,
        env=os.environ.copy(),
    )

print(
    f"\nProject0 process started "
    f"(PID {dashboard.pid})."
)
print("Waiting for Dashboard...")


# ============================================================
# Wait for the Dashboard.
# ============================================================

dashboard_ready = False
deadline = (
    time.monotonic()
    + DASHBOARD_STARTUP_TIMEOUT_SECONDS
)

while time.monotonic() < deadline:
    if dashboard.poll() is not None:
        print(
            "Project0 Dashboard process exited "
            f"with code {dashboard.returncode}."
        )
        break

    try:
        with urllib.request.urlopen(
            DASHBOARD_URL,
            timeout=2,
        ) as response:
            if response.status == 200:
                print("Project0 Dashboard: RUNNING")
                print("HTTP status:", response.status)

                dashboard_ready = True
                break

    except Exception:
        pass

    time.sleep(1)


# ============================================================
# Stop immediately and show diagnostics on failure.
# ============================================================

if not dashboard_ready:
    print("Project0 Dashboard did not start.")
    print("\n===== Project0 log =====")

    log_text = PROJECT0_LOG.read_text(
        errors="replace"
    )
    print(log_text or "(Dashboard log is empty.)")

    if dashboard.poll() is None:
        dashboard.terminate()

        try:
            dashboard.wait(timeout=5)
        except subprocess.TimeoutExpired:
            dashboard.kill()
            dashboard.wait()

    raise RuntimeError(
        "Project0 Dashboard failed to start within "
        f"{DASHBOARD_STARTUP_TIMEOUT_SECONDS} seconds. "
        "Review the log above, correct the error, "
        "and rerun Step 6."
    )



## Step 7 - Open the Project0 Dashboard

The Dashboard runs inside Colab at the private address `127.0.0.1:8001`, which your browser cannot normally access. This step first confirms that the Dashboard is responding, then installs Cloudflare's `cloudflared` utility when necessary and creates a temporary encrypted tunnel to the local Dashboard. No Cloudflare account or credentials are required.

If the Dashboard is not ready, this step stops immediately and instructs you to run Step 6 successfully. Tunnel startup is limited to 45 seconds. If you rerun this step, it stops the tunnel previously created by the notebook before opening a new one.

When startup succeeds, Colab displays an **Open Project0 Dashboard** link. Open it in a new browser tab.

The generated `trycloudflare.com` URL:

- Exists only while the Colab runtime and tunnel process remain active.
- Changes whenever a new tunnel is created.
- Is intended for a short-lived, single-user session.
- Grants anyone who has the URL access to the active Dashboard, so do not share it.
- May allow changes within the disposable `/content` workspace, but does not by itself grant permission to modify the remote GitHub repository.

Do not submit confidential information through the temporary tunnel. When finished, run the cleanup step to stop the tunnel and Dashboard processes.

The tunnel carries only Dashboard browser traffic. Ollama continues to run privately inside the Colab runtime.

In [ ]:
from pathlib import Path
import re
import selectors
import subprocess
import time
import urllib.request
from IPython.display import display, HTML

CLOUDFLARED_PATH = Path("/usr/local/bin/cloudflared")
DASHBOARD_URL = "http://127.0.0.1:8001"
TUNNEL_STARTUP_TIMEOUT_SECONDS = 45


def stop_process(process, timeout=5):
    """Terminate a process, escalating to kill when necessary."""

    if process is None or process.poll() is not None:
        return

    process.terminate()

    try:
        process.wait(timeout=timeout)
    except subprocess.TimeoutExpired:
        process.kill()
        process.wait()


# ---------------------------------------------------------------------
# Verify that Step 6 completed successfully.
# ---------------------------------------------------------------------

try:
    with urllib.request.urlopen(
        DASHBOARD_URL,
        timeout=3,
    ) as response:
        if response.status != 200:
            raise RuntimeError(
                "Project0 Dashboard returned unexpected "
                f"HTTP status {response.status}."
            )
except Exception as error:
    raise RuntimeError(
        "Project0 Dashboard is not ready. "
        "Run Step 6 successfully before starting the tunnel."
    ) from error

print("Project0 Dashboard readiness: CONFIRMED")


# ---------------------------------------------------------------------
# Install cloudflared if necessary.
# ---------------------------------------------------------------------

if not CLOUDFLARED_PATH.exists():
    print("Installing cloudflared...")

    subprocess.run(
        [
            "wget",
            "-q",
            "https://github.com/cloudflare/cloudflared/"
            "releases/latest/download/cloudflared-linux-amd64",
            "-O",
            str(CLOUDFLARED_PATH),
        ],
        check=True,
    )

    subprocess.run(
        ["chmod", "+x", str(CLOUDFLARED_PATH)],
        check=True,
    )

print(
    subprocess.run(
        [str(CLOUDFLARED_PATH), "--version"],
        check=True,
        capture_output=True,
        text=True,
    ).stdout.strip()
)


# ---------------------------------------------------------------------
# Stop a tunnel previously started by this cell.
# ---------------------------------------------------------------------

existing_tunnel = globals().get("cloudflared_process")

if existing_tunnel is not None and existing_tunnel.poll() is None:
    print("Stopping existing Project0 tunnel...")
    stop_process(existing_tunnel)


# ---------------------------------------------------------------------
# Start a Cloudflare Quick Tunnel.
# ---------------------------------------------------------------------

print("Starting Project0 Cloudflare tunnel...")

cloudflared_process = subprocess.Popen(
    [
        str(CLOUDFLARED_PATH),
        "tunnel",
        "--url",
        DASHBOARD_URL,
        "--no-autoupdate",
    ],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

if cloudflared_process.stdout is None:
    stop_process(cloudflared_process)

    raise RuntimeError(
        "Cloudflare tunnel output could not be captured."
    )

selector = selectors.DefaultSelector()
selector.register(
    cloudflared_process.stdout,
    selectors.EVENT_READ,
)

tunnel_url = None
deadline = time.monotonic() + TUNNEL_STARTUP_TIMEOUT_SECONDS

try:
    while True:
        remaining = deadline - time.monotonic()

        if remaining <= 0:
            break

        events = selector.select(timeout=min(remaining, 1.0))

        for key, _ in events:
            line = key.fileobj.readline()

            if not line:
                continue

            print(line.rstrip())

            match = re.search(
                r"https://[a-zA-Z0-9-]+\.trycloudflare\.com",
                line,
            )

            if match:
                tunnel_url = match.group(0)
                break

        if tunnel_url is not None:
            break

        if cloudflared_process.poll() is not None:
            break
finally:
    selector.close()


# ---------------------------------------------------------------------
# Fail cleanly if the tunnel did not start.
# ---------------------------------------------------------------------

if tunnel_url is None:
    return_code = cloudflared_process.poll()
    stop_process(cloudflared_process)

    if return_code is None:
        reason = (
            "Cloudflare did not provide a URL within "
            f"{TUNNEL_STARTUP_TIMEOUT_SECONDS} seconds."
        )
    else:
        reason = (
            "Cloudflare exited before providing a URL "
            f"(exit code {return_code})."
        )

    raise RuntimeError(
        f"{reason} Verify that Project0 is running at "
        f"{DASHBOARD_URL} and rerun this cell."
    )


# ---------------------------------------------------------------------
# Display the Dashboard link.
# ---------------------------------------------------------------------

print()
print("Project0 Dashboard ready:")
print(tunnel_url)
print(
    "This is a short-lived, single-user URL. "
    "Do not share it or submit sensitive information."
)

display(
    HTML(
        f"""
        <p>
          <a href="{tunnel_url}"
             target="_blank"
             rel="noopener noreferrer"
             style="font-size:18px;font-weight:bold;">
            Open Project0 Dashboard
          </a>
        </p>
        """
    )
)



## Step 8 - Stop Project0 (Optional)

Use this optional cleanup step when you are finished with the Project0 Dashboard.

The `STOP_PROJECT0` checkbox in the following cell defaults to **False**. Therefore, **Run all** safely skips cleanup and leaves both the Dashboard and its public link running. The message `Project0 cleanup skipped.` confirms this behavior.

When you are ready to stop the services:

1. Enable the `STOP_PROJECT0` checkbox.
2. Run the Step 8 code cell manually.

The cleanup stops the Cloudflare Quick Tunnel created in Step 7 and the Project0 Dashboard process created in Step 6. It does not delete the cloned repository, downloaded model, logs, or other files in the current runtime.

To end the entire Colab session and remove all temporary files under `/content`, use **Runtime → Disconnect and delete runtime**.

In [ ]:
import subprocess

STOP_PROJECT0 = False  # @param {type:"boolean"}

if not STOP_PROJECT0:
    print("Project0 cleanup skipped.")
else:
    # Stop the Cloudflare Quick Tunnel.
    if (
        "cloudflared_process" in globals()
        and cloudflared_process.poll() is None
    ):
        print("Stopping Project0 Cloudflare tunnel...")
        cloudflared_process.terminate()

        try:
            cloudflared_process.wait(timeout=5)
        except subprocess.TimeoutExpired:
            print(
                "Cloudflare tunnel did not stop normally; "
                "terminating it..."
            )
            cloudflared_process.kill()
            cloudflared_process.wait()

        print("Cloudflare tunnel stopped.")
    else:
        print("Cloudflare tunnel is not running.")

    # Stop the Project0 Dashboard.
    if "dashboard" in globals() and dashboard.poll() is None:
        print("Stopping Project0 Dashboard...")
        dashboard.terminate()

        try:
            dashboard.wait(timeout=10)
        except subprocess.TimeoutExpired:
            print(
                "Project0 did not stop normally; terminating it..."
            )
            dashboard.kill()
            dashboard.wait()

        print("Project0 Dashboard stopped.")
    else:
        print("Project0 Dashboard is not running.")

    print()
    print("Project0 Colab session cleanup complete.")

